# Step 24 — Does the metric see a worse model? And where is class mass set?

**Inputs.** Mount the committed outputs of **Step 19** (reference logits, classifier) and **Step 21** (`step21_baseline.npz`, `step21_mass_correct.npz`). GPU and Internet on. Committed run, about 6.5 hours.

**A. Power.** Two kinds of graded degradation, both keeping the 256-step sampler and its recording grid:
- *Score scale* $\lambda\in\{0.995,0.98,0.95\}$: the learned score is shrunk at every noise level.
- *Noise-conditioning mismatch* $\kappa\in\{1.1,1.25\}$: the denoiser is told the noise level is $\kappa\sigma$ instead of $\sigma$, a realistic kind of mis-specification.

For each, transitions are compared **pair by pair against Step 21's unguided baseline**: same sampler, same $n$, bootstrap medians, class-block intervals on the median offset. We report the raw and rebalanced offsets, the rebalanced asymmetry agreement with the data, class-frequency error and FID. A useful metric should move monotonically with the degradation.

A step-count sweep is not included. Coarser samplers do not visit the fixed recording levels, so their transitions could not be compared on the same grid.

**B. Where class mass is set.** The exact class-reweighting tilt of Step 21 is applied only at high noise ($\sigma\ge2.4$, where Step 23 saw superclass decisions forming) or only at low noise ($\sigma\le2.4$). For each, the share of the full-range correction it achieves.

In [ ]:
import os; os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
import sys, subprocess, pickle, glob, math, time, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from itertools import combinations
from torchvision import datasets
from scipy.signal import fftconvolve
dev = 'cuda'; torch.manual_seed(0); rng = np.random.default_rng(0); plt.rcParams['figure.dpi'] = 120
N_COND = 2500            # trajectories per condition (EM-256); lower if the first-chunk estimate is too long
CHUNK = 500
CLASSES = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']; VEH = {0, 1, 8, 9}; PAIRS = list(combinations(range(10), 2))
P19 = glob.glob('/kaggle/input/**/step19_logits.npz', recursive=True); assert P19, 'mount the Step 19 output'
Z19 = np.load(P19[0]); L_TE, W_TE, L_M0, W_M0, SIG_REC = Z19['L_TE'], Z19['W_TE'], Z19['L_M'], Z19['W_M'], Z19['SIG_REC']
SIGMA_MAX, SIGMA_MIN, RHO, STEPS = 80.0, 0.05, 7.0, 256
def edm_sched(steps=STEPS, s_max=SIGMA_MAX, s_min=SIGMA_MIN, rho=RHO):
    i = np.arange(steps); return (s_max**(1/rho) + i/(steps-1)*(s_min**(1/rho) - s_max**(1/rho)))**rho
SCHED = edm_sched(); REC = {int(np.argmin(np.abs(SCHED - s))): gi for gi, s in enumerate(SIG_REC)}
assert np.allclose(SCHED[sorted(REC)], np.sort(SIG_REC)[::-1]), 'recording grid does not match Step 19'
print('reference', L_TE.shape, 'Step 19 baseline trajectories', L_M0.shape)

# ---- classifier used for the reference (from Step 19) ----
class FiLMBlock(nn.Module):
    def __init__(self, cin, cout, emb, stride=1):
        super().__init__(); self.c1 = nn.Conv2d(cin, cout, 3, stride, 1, bias=False); self.b1 = nn.BatchNorm2d(cout); self.c2 = nn.Conv2d(cout, cout, 3, 1, 1, bias=False); self.b2 = nn.BatchNorm2d(cout)
        self.film = nn.Linear(emb, 2*cout); self.sc = nn.Sequential(nn.Conv2d(cin, cout, 1, stride, bias=False), nn.BatchNorm2d(cout)) if (stride != 1 or cin != cout) else nn.Identity()
    def forward(self, x, e):
        g, b = self.film(e).chunk(2, 1); h = F.silu(self.b1(self.c1(x))); h = self.b2(self.c2(h))*(1+g[:, :, None, None]) + b[:, :, None, None]; return F.silu(h + self.sc(x))
class NoisyResNet(nn.Module):
    def __init__(self, ch=64, emb=128):
        super().__init__(); self.emb = nn.Sequential(nn.Linear(32, emb), nn.SiLU(), nn.Linear(emb, emb)); self.inp = nn.Conv2d(3, ch, 3, 1, 1)
        self.blocks = nn.ModuleList([FiLMBlock(ch, ch, emb), FiLMBlock(ch, 2*ch, emb, 2), FiLMBlock(2*ch, 2*ch, emb), FiLMBlock(2*ch, 4*ch, emb, 2), FiLMBlock(4*ch, 4*ch, emb)]); self.fc = nn.Linear(4*ch, 10)
    def forward(self, x, sigma):
        f = torch.exp(torch.arange(16, device=x.device)*-(math.log(100)/15)); t = 0.25*torch.log(sigma)[:, None]*f; e = self.emb(torch.cat([torch.sin(t), torch.cos(t)], 1)); h = self.inp(x)
        for b in self.blocks: h = b(h, e)
        return self.fc(h.mean((2, 3)))
CLF = NoisyResNet().to(dev); CLF.load_state_dict(torch.load(glob.glob('/kaggle/input/**/noisy_resnet_seed0.pt', recursive=True)[0], map_location=dev)); CLF.eval()
for p in CLF.parameters(): p.requires_grad_(False)
@torch.no_grad()
def clf_logits(x, sigma, bs=1000):
    out = []
    for xb in x.split(bs):
        with torch.autocast('cuda', dtype=torch.float16): out.append(CLF(xb, torch.full((len(xb),), float(sigma), device=dev)).float())
    return torch.cat(out)

# ---- pretrained EDM ----
if not os.path.exists('/kaggle/working/edm'): subprocess.run(['git', 'clone', '-q', 'https://github.com/NVlabs/edm.git', '/kaggle/working/edm'], check=True)
sys.path.insert(0, '/kaggle/working/edm'); import dnnlib
with dnnlib.util.open_url('https://nvlabs-fi-cdn.nvidia.com/edm/pretrained/edm-cifar10-32x32-uncond-vp.pkl') as f: NET = pickle.load(f)['ema'].to(dev).eval()
NET.use_fp16 = True
ACTIVE = {'net': None, 'labels': None, 'kappa': 1.0}
@torch.no_grad()
def D(x, s):
    if ACTIVE['net'] is None: return NET(x, torch.full((len(x),), float(s)*ACTIVE['kappa'], device=dev)).float()
    return ACTIVE['net'](x, torch.full((len(x),), float(s), device=dev), ACTIVE['labels'][:len(x)]).float()
# reference class frequencies (the classifier's labels on test data) = mass-correction target
F_REF = np.bincount(W_TE.argmax(1), minlength=10)/len(W_TE); F_BASE19 = np.bincount(W_M0.argmax(1), minlength=10)/len(W_M0)
print('classifier label frequency on test:', np.round(F_REF, 3)); print('Step 19 generated frequency:     ', np.round(F_BASE19, 3))

In [ ]:
def guide_grad(x, s, logr):
    """Exact class-reweighting tilt: grad_x [ logsumexp(l + log r) - logsumexp(l) ] with the noise-conditional classifier."""
    with torch.enable_grad():
        xg = x.detach().requires_grad_(True); l = CLF(xg, torch.full((len(xg),), float(s), device=dev)).float()
        obj = (torch.logsumexp(l + logr, 1) - torch.logsumexp(l, 1)).sum(); return torch.autograd.grad(obj, xg)[0]
@torch.no_grad()
def run_sde(n, seed, logr=None, window=None, lam=1.0, keep_states_at=None):
    """Reverse SDE (Euler-Maruyama, 256 steps) as in Step 19. Returns logits at the recorded levels, terminal weights, final denoised images
    (uint8), and optionally the states at given step indices (for branching)."""
    g = torch.Generator(device=dev).manual_seed(seed); x = torch.randn(n, 3, 32, 32, generator=g, device=dev)*SCHED[0]
    Lrec = np.zeros((len(SIG_REC), n, 10), np.float16); states = {}
    lr = None if logr is None else torch.as_tensor(logr, dtype=torch.float32, device=dev)
    for i in range(STEPS):
        s0 = float(SCHED[i]); s1 = float(SCHED[i+1]) if i+1 < STEPS else SIGMA_MIN
        if i in REC: Lrec[REC[i]] = clf_logits(x, s0).cpu().numpy().astype(np.float16)
        if keep_states_at is not None and i in keep_states_at: states[i] = x.clone()
        if s1 >= s0: continue
        d = D(x, s0); sc = lam*(d - x)/s0**2
        if lr is not None and (window is None or window[0] <= s0 <= window[1]): sc = sc + guide_grad(x, s0, lr)
        x = x + (s1 - s0)*(-2*s0*sc) + math.sqrt(2*s0*(s0 - s1))*torch.randn(x.shape, generator=g, device=dev)
    W = F.softmax(clf_logits(x, SIGMA_MIN), 1).cpu().numpy(); x0 = D(x, SIGMA_MIN).clamp(-1, 1)
    img = ((x0 + 1)*127.5).round().to(torch.uint8).cpu().numpy()
    return Lrec, W, img, states
def run_condition(name, n, seed0, **kw):
    t0 = time.time(); Ls, Ws, Is = [], [], []
    for c in range(0, n, CHUNK):
        L_, W_, I_, _ = run_sde(min(CHUNK, n - c), seed0 + c, **kw); Ls.append(L_); Ws.append(W_); Is.append(I_)
        if c == 0: print(f"  {name}: first chunk {time.time()-t0:.0f}s -> estimated {(time.time()-t0)*math.ceil(n/CHUNK)/60:.0f} min")
    L_, W_, I_ = np.concatenate(Ls, 1), np.concatenate(Ws), np.concatenate(Is)
    np.savez_compressed(f'/kaggle/working/step24_{name}.npz', L=L_, W=W_, IMG=I_); print(f"  {name}: {len(W_)} trajectories ({(time.time()-t0)/60:.0f} min)"); return L_, W_, I_

def to_float(img): return torch.from_numpy(img).float().div(127.5).sub(1).to(dev)
@torch.no_grad()
def our_labels(img): return np.concatenate([clf_logits(to_float(img[i:i+1000]) + SIGMA_MIN*torch.randn(len(img[i:i+1000]), 3, 32, 32, device=dev), SIGMA_MIN).argmax(1).cpu().numpy() for i in range(0, len(img), 1000)])
# independent strong classifier: ViT fine-tuned on CIFAR-10 (HuggingFace)
VIT = None
try:
    from transformers import AutoModelForImageClassification
    VIT = AutoModelForImageClassification.from_pretrained('aaraki/vit-base-patch16-224-in21k-finetuned-cifar10').to(dev).eval().half()
    names = [VIT.config.id2label[i].lower() for i in range(10)]; VMAP = np.array([CLASSES.index(n) if n in CLASSES else i for i, n in enumerate(names)]); print('ViT labels:', names)
except Exception as e: print('ViT unavailable:', e)
@torch.no_grad()
def vit_labels(img, bs=128):
    out = []
    for i in range(0, len(img), bs):
        x = to_float(img[i:i+bs]); x = F.interpolate(x, size=224, mode='bilinear', align_corners=False)          # [-1,1] == (x-0.5)/0.5 normalisation
        out.append(VIT(pixel_values=x.half()).logits.argmax(1).cpu().numpy())
    return VMAP[np.concatenate(out)]
dst = datasets.CIFAR10('/kaggle/working/cifar', train=False, download=True); TEST_IMG = np.asarray(dst.data).transpose(0, 3, 1, 2); YTE = np.asarray(dst.targets)
if VIT is not None: print(f"ViT test accuracy (2000 images): {np.mean(vit_labels(TEST_IMG[:2000]) == YTE[:2000]):.3f}")
print(f"our classifier test accuracy (sigma_min): {np.mean(our_labels(TEST_IMG[:2000]) == YTE[:2000]):.3f}")
# FID (pytorch-fid Inception) against the 50k training images

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pytorch-fid'], check=True)
from pytorch_fid.inception import InceptionV3; from pytorch_fid.fid_score import calculate_frechet_distance
INC = InceptionV3([InceptionV3.BLOCK_INDEX_BY_DIM[2048]]).to(dev).eval()
@torch.no_grad()
def inc_stats(img, bs=250):
    f = np.concatenate([INC(torch.from_numpy(img[i:i+bs]).float().div(255).to(dev))[0].squeeze(-1).squeeze(-1).cpu().numpy() for i in range(0, len(img), bs)]); return f.mean(0), np.cov(f, rowvar=False)
dtr = datasets.CIFAR10('/kaggle/working/cifar', train=True, download=True); MU_T, S_T = inc_stats(np.asarray(dtr.data).transpose(0, 3, 1, 2))
fid = lambda img: calculate_frechet_distance(*inc_stats(img), MU_T, S_T)

def freq(lbl): return np.bincount(lbl, minlength=10)/len(lbl)
def find(pat): p = glob.glob(f'/kaggle/input/**/{pat}', recursive=True); assert p, f'mount the output containing {pat}'; return p[0]
B0 = np.load(find('step21_baseline.npz')); BASE = (B0['L'][:, :N_COND], B0['W'][:N_COND], B0['IMG'][:N_COND])
M1 = np.load(find('step21_mass_correct.npz')); MC_FULL = (M1['L'][:, :N_COND], M1['W'][:N_COND], M1['IMG'][:N_COND])
LOGR_MASS = np.log(F_REF/F_BASE19); print('baseline and full-range mass correction loaded:', BASE[1].shape, MC_FULL[1].shape)

## Sampling the conditions

In [ ]:
CONDS = {}
for lam in [0.995, 0.98, 0.95]: CONDS[f'scale_{lam}'] = run_condition(f'scale_{lam}', N_COND, 81000 + int(1000*lam), lam=lam)
for kap in [1.1, 1.25]:
    ACTIVE['kappa'] = kap; CONDS[f'mismatch_{kap}'] = run_condition(f'mismatch_{kap}', N_COND, 82000 + int(100*kap)); ACTIVE['kappa'] = 1.0
CONDS['mass_high_noise'] = run_condition('mass_high_noise', N_COND, 83000, logr=LOGR_MASS, window=(2.4, 1e9))
CONDS['mass_low_noise'] = run_condition('mass_low_noise', N_COND, 84000, logr=LOGR_MASS, window=(0.0, 2.4))

## Analysis

In [ ]:
G = np.linspace(-4.0, 4.0, 4001); DGG = G[1]-G[0]
def wkde_score(m, w):
    m = np.clip(m, -8, 8); w = np.asarray(w, dtype=np.float64); neff = w.sum()**2/np.maximum((w**2).sum(), 1e-12)
    mu = np.average(m, weights=w); sd = np.sqrt(np.average((m-mu)**2, weights=w)); bw = max(1.5*1.06*sd*neff**(-1/5), 1e-3)
    h, _ = np.histogram(m, bins=np.append(G-DGG/2, G[-1]+DGG/2), weights=w, density=True); k = np.exp(-G**2/(2*bw**2)); k /= k.sum()
    p = np.maximum(fftconvolve(h, k, 'same'), 1e-300); return np.gradient(np.log(p), G), p
def _merge(pts, merge=0.03):
    ch = True
    while ch and len(pts) > 1:
        ch = False
        for a in range(len(pts)-1):
            if abs(pts[a][0]-pts[a+1][0]) < merge and pts[a][1] != pts[a+1][1]: del pts[a:a+2]; ch = True; break
    return pts
def stationary(sm, p, lo=-3.5, hi=3.5, prominence=0.05):
    keep = (G > lo) & (G < hi) & (p > 1e-3*p.max()); idx = np.where((np.sign(sm[:-1]) != np.sign(sm[1:])) & keep[:-1] & keep[1:])[0]
    pts = _merge([(float(G[i]-sm[i]*DGG/(sm[i+1]-sm[i])) if sm[i+1] != sm[i] else float(G[i]), 'max' if sm[i] > 0 else 'min') for i in idx])
    logp = np.concatenate([[0.0], np.cumsum(0.5*(sm[1:]+sm[:-1])*DGG)]); lp = lambda x: float(np.interp(x, G, logp)); ch = True
    while ch and len(pts) > 1:
        ch = False
        for a in range(len(pts)-1):
            if pts[a][1] != pts[a+1][1] and abs(lp(pts[a][0]) - lp(pts[a+1][0])) < prominence: del pts[a:a+2]; ch = True; break
    return pts
def n_max(pts): return sum(k == 'max' for _, k in pts)
def cusp_h(sm, p, window=1.0, b_min=0.02, s_max=0.6):
    ok = (np.abs(G) < window) & (p > 0); c3, c2, c1, c0 = np.polyfit(G[ok], sm[ok], 3, w=np.sqrt(p[ok])); b = max(-c3, b_min); c3 = -b; s = float(np.clip(-c2/(3*c3), -s_max, s_max))
    return c0 + c1*s + c2*s**2 + c3*s**3
def std_stats(z, wa, wb):
    mua, mub = np.average(z, weights=wa), np.average(z, weights=wb); half = 0.5*(mua-mub); sda = np.sqrt(np.average((z-mua)**2, weights=wa)); sdb = np.sqrt(np.average((z-mub)**2, weights=wb))
    return 0.5*(mua+mub), half, abs(half)/(0.5*(sda+sdb))
STD = {(a, b): [std_stats(L_TE[g, :, a].astype(np.float64) - L_TE[g, :, b], W_TE[:, a] + 1e-12, W_TE[:, b] + 1e-12) for g in range(len(SIG_REC))] for a, b in PAIRS}
def pair_sh(L, W, a, b, idx, balance=False):
    """(sigma_bif, h) along the pair log-odds, standardised by the reference, weighted by terminal class weights."""
    Wa = W[idx, a].astype(np.float64); Wb = W[idx, b].astype(np.float64)
    if balance: tot = 0.5*(Wa.sum() + Wb.sum()); Wa = Wa/Wa.sum()*tot; Wb = Wb/Wb.sum()*tot
    prev = None
    for g in range(len(SIG_REC)):
        mid, half, sep = STD[(a, b)][g]
        if sep < 0.3: prev = 1; continue
        z = L[g, idx, a].astype(np.float64) - L[g, idx, b]; sm, p = wkde_score((z - mid)/half, Wa + Wb); nm = n_max(stationary(sm, p))
        if nm >= 2 and prev is not None: return float(SIG_REC[g]), cusp_h(sm, p)
        if nm == 1: prev = 1
    return None, None
def evaluate(L, W, name, NB=40):
    n = len(W); ir = rng.choice(len(W_TE), size=n, replace=False); allm = np.arange(n); rows = []
    fm = np.bincount(W.argmax(1), minlength=10)/n; over = np.log(fm/F_REF)
    for a, b in PAIRS:
        sm_, hm = pair_sh(L, W, a, b, allm); sr, hr = pair_sh(L_TE, W_TE, a, b, ir); _, hmb = pair_sh(L, W, a, b, allm, balance=True)
        if sm_ is None or sr is None: continue
        rat = []
        for _ in range(NB):
            s1, _ = pair_sh(L, W, a, b, rng.integers(n, size=n)); s2, _ = pair_sh(L_TE, W_TE, a, b, rng.choice(len(W_TE), size=n))
            if s1 and s2: rat.append(s1/s2)
        lo, hi = np.percentile(rat, [2.5, 97.5]) if len(rat) > 10 else (np.nan, np.nan)
        rows.append(dict(pair=(a, b), sm=sm_, sr=sr, hm=hm, hr=hr, hmb=hmb, lo=lo, hi=hi, dover=over[a]-over[b]))
    R = np.array([r['sm']/r['sr'] for r in rows]); hm = np.array([r['hm'] for r in rows]); hr = np.array([r['hr'] for r in rows]); hmb = np.array([r['hmb'] for r in rows]); dov = np.array([r['dover'] for r in rows])
    lo = np.array([r['lo'] for r in rows]); hi = np.array([r['hi'] for r in rows])
    print(f"{name:>20}: pairs {len(rows)}; sigma_bif ratio median {np.median(R):.2f} (CI>1: {(lo>1).sum()}, CI<1: {(hi<1).sum()}); corr h model-data raw {np.corrcoef(hm, hr)[0,1]:.2f}, rebalanced {np.corrcoef(hmb, hr)[0,1]:.2f}; "
          f"corr h model vs over-generation {np.corrcoef(hm, dov)[0,1]:.2f}; mean |log over-generation| {np.mean(np.abs(over)):.3f}")
    return {r['pair']: r for r in rows}, fm

from itertools import combinations as comb
def boot_sb(L, W, a, b, n, B=30, balance=False):
    """bootstrap medians of (sigma_bif, h) with optional class-mass rebalancing"""
    def one(idx, g0=0):
        Wa = W[idx, a].astype(np.float64); Wb = W[idx, b].astype(np.float64)
        if balance: tot = 0.5*(Wa.sum()+Wb.sum()); Wa = Wa/Wa.sum()*tot; Wb = Wb/Wb.sum()*tot
        prev = None if g0 == 0 else 1
        for g in range(g0, len(SIG_REC)):
            mid, half, sep = STD[(a, b)][g]
            if sep < 0.3: prev = 1; continue
            z = L[g, idx, a].astype(np.float64) - L[g, idx, b]; sm, p = wkde_score((z - mid)/half, Wa + Wb); nm = n_max(stationary(sm, p))
            if nm >= 2 and prev is not None: return float(SIG_REC[g]), cusp_h(sm, p), g
            if nm == 1: prev = 1
        return None
    r0 = one(rng.choice(len(W), size=n, replace=len(W) < n))
    if r0 is None: return None
    out = [one(rng.choice(len(W), size=n, replace=True), max(0, r0[2] - 8)) for _ in range(B)]; out = [o for o in out if o]
    return (np.median([o[0] for o in out]), np.median([o[1] for o in out])) if len(out) >= B//3 else None
def class_block(lr, P, B=3000):
    P = np.array(P); vals = []
    for _ in range(B):
        cls = rng.choice(10, 10, replace=True); w = np.array([np.sum(cls == a)*np.sum(cls == b) for a, b in P], float)
        if w.sum() >= 6: vals.append(np.exp(np.average(lr, weights=w)))
    return np.percentile(vals, [2.5, 97.5])
INC_STATS_DONE = True
n = N_COND; REF = {}; BASES = {}
for p in PAIRS:
    r = boot_sb(L_TE, W_TE, *p, n); rb = boot_sb(BASE[0], BASE[1], *p, n); rbb = boot_sb(BASE[0], BASE[1], *p, n, balance=True)
    if r and rb and rbb: REF[p] = r; BASES[p] = (rb, rbb)
print(f"reference and baseline: {len(REF)} pairs")
def report(name, L, W, IMG):
    rows = {}
    for p in REF:
        x = boot_sb(L, W, *p, n); xb = boot_sb(L, W, *p, n, balance=True)
        if x and xb: rows[p] = (x, xb)
    P = list(rows)
    lr_raw = np.array([np.log(rows[p][0][0]/BASES[p][0][0]) for p in P]); lr_rb = np.array([np.log(rows[p][1][0]/BASES[p][1][0]) for p in P])
    h_rb = np.array([rows[p][1][1] for p in P]); h_ref = np.array([REF[p][1] for p in P]); h_raw = np.array([rows[p][0][1] for p in P])
    fo = freq(W.argmax(1)); fv = freq(vit_labels(IMG)) if VIT is not None else fo
    ci_raw, ci_rb = class_block(lr_raw, P), class_block(lr_rb, P)
    out = dict(n_pairs=len(P), raw=np.exp(np.mean(lr_raw)), raw_ci=ci_raw, rb=np.exp(np.mean(lr_rb)), rb_ci=ci_rb, sd_rb=lr_rb.std(),
               h_rb=np.corrcoef(h_rb, h_ref)[0, 1], h_raw=np.corrcoef(h_raw, h_ref)[0, 1], fdev=np.mean(np.abs(np.log(fo/F_REF))), fdev_vit=np.mean(np.abs(np.log(fv/0.1))), fid=fid(IMG))
    print(f"{name:>18}: pairs {out['n_pairs']}; sigma_bif vs baseline raw x{out['raw']:.3f} [{ci_raw[0]:.3f}, {ci_raw[1]:.3f}], rebalanced x{out['rb']:.3f} [{ci_rb[0]:.3f}, {ci_rb[1]:.3f}]; "
          f"rebalanced h vs data {out['h_rb']:.2f} (raw {out['h_raw']:.2f}); class-freq error {out['fdev']:.3f} (ViT {out['fdev_vit']:.3f}); FID {out['fid']:.2f}")
    return out
t0 = time.time(); RES = {'baseline': report('baseline', *BASE)}
RES['mass_full (Step 21)'] = report('mass_full (Step 21)', *MC_FULL)
for k, v in CONDS.items(): RES[k] = report(k, *v)
print(f"({(time.time()-t0)/60:.0f} min)")

In [ ]:
fb, ff = RES['baseline']['fdev'], RES['mass_full (Step 21)']['fdev']
for k in ['mass_high_noise', 'mass_low_noise']:
    print(f"{k:>16}: class-frequency error {RES[k]['fdev']:.3f} (baseline {fb:.3f}, full-range correction {ff:.3f}) -> {100*(fb - RES[k]['fdev'])/(fb - ff):.0f}% of the full correction; "
          f"raw h vs data {RES[k]['h_raw']:.2f}; FID {RES[k]['fid']:.2f}")
fig, ax = plt.subplots(1, 3, figsize=(13, 3.6))
lams = [1.0, 0.995, 0.98, 0.95]; kaps = [1.0, 1.1, 1.25]; keyL = ['baseline'] + [f'scale_{l}' for l in lams[1:]]; keyK = ['baseline'] + [f'mismatch_{k}' for k in kaps[1:]]
for ks, xs, lab, c in [(keyL, lams, 'score scale lambda', 'C0'), (keyK, kaps, 'conditioning mismatch kappa', 'C1')]:
    rb = [RES[k]['rb'] for k in ks]; lo = [RES[k]['rb_ci'][0] if k != 'baseline' else 1 for k in ks]; hi = [RES[k]['rb_ci'][1] if k != 'baseline' else 1 for k in ks]
    ax[0].errorbar(range(len(ks)), rb, yerr=[np.clip(np.array(rb)-lo, 0, None), np.clip(np.array(hi)-rb, 0, None)], fmt='o-', color=c, label=lab, capsize=2)
    ax[1].plot(range(len(ks)), [RES[k]['h_rb'] for k in ks], 'o-', color=c, label=lab); ax[2].plot(range(len(ks)), [RES[k]['fid'] for k in ks], 'o-', color=c, label=lab)
ax[0].axhline(1, color='k', lw=0.5); ax[0].set_ylabel('sigma_bif vs baseline (rebalanced, geometric mean)'); ax[1].set_ylabel('rebalanced h vs data (corr)'); ax[2].set_ylabel('FID (n=%d)' % N_COND)
for a_ in ax: a_.set_xlabel('degradation level (0 = baseline)'); a_.legend(fontsize=7)
plt.tight_layout(); plt.show()

## What to report back
- The baseline and mass_full lines.
- One line per condition.
- The two band lines.
- The three panels.

A good metric moves monotonically with $\lambda$ and $\kappa$, with intervals that exclude 1, ideally before FID changes much.